# Participants Preprocessing

Merges prescreening data with session data and computes volume in dB SPL.

**Sources (in `demographics/`):**
- `1_prescreened.xlsx` — token, age, sex, gender
- `3_data_summary_active.csv` — token, participant_ID, group, sound loudness
- `3_data_summary_passive.csv` — same

**Output on NAS:**
- `{date}_active_demographics.csv`
- `{date}_passive_demographics.csv`


In [ ]:
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

DATE_STR   = datetime.today().strftime("%Y%m%d")
DEMOGRAPHICS_DIR = Path("L:/EEG_Flow_data/participant_records/demographics")
DEMOGRAPHICS_ACTIVE_DIR = Path("L:/EEG_Flow_data/Study_EEG_Flow_Active/raw/participant_records/experiment_log")
DEMOGRAPHICS_PASSIVE_DIR = Path("L:/EEG_Flow_data/Study_EEG_Flow_Passive/raw/participant_records/experiment_log")

if __import__('sys').platform == "win32":
    OUT_ACTIVE  = Path("L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior")
    OUT_PASSIVE = Path("L:/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/outputs/measures/behavior")
else:
    OUT_ACTIVE  = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior")
    OUT_PASSIVE = Path("/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/outputs/measures/behavior")

OUT_ACTIVE.mkdir(parents=True, exist_ok=True)
OUT_PASSIVE.mkdir(parents=True, exist_ok=True)
print("Config OK")


In [ ]:
# ==============================================================================
# Load demographics files
# ==============================================================================

# Prescreening
df_pre = pd.read_excel(DEMOGRAPHICS_DIR / "1_prescreened.xlsx")
print(f"Prescreened: {len(df_pre)} rows | columns: {df_pre.columns.tolist()}\n")

# Session data (Qualtrics double header — adjust header= if needed)
df_active  = pd.read_excel(DEMOGRAPHICS_ACTIVE_DIR / "3_data_summary_active.xlsx",  header=1)
df_passive = pd.read_excel(DEMOGRAPHICS_PASSIVE_DIR / "3_data_summary_passive.xlsx", header=1)
print(f"Active  session: {len(df_active)}  rows | columns: {df_active.columns.tolist()}")
print(f"Passive session: {len(df_passive)} rows | columns: {df_passive.columns.tolist()}")


In [ ]:
# ==============================================================================
# Rename columns — adapt to your actual column names
# ==============================================================================

df_pre = df_pre.rename(columns={
    "id":     "token",
    "age":    "age",
    "sex":    "sex",
})[["token", "age", "sex"]]

for df in [df_active, df_passive]:
    df.rename(columns={
        "token":          "token",
        "Participant ID": "participant",
        "Group":          "group",
        "sound loudness": "volume_pct",
    }, inplace=True)

    df['lvl_highflow'] = np.where(
    df['cond_UT1'] == 'High Flow',
    df['lvl_UT1'],
    df['lvl_UT2']
)

df_active  = df_active[["token",  "participant", "group", "volume_pct", "lvl_highflow"]]
df_passive = df_passive[["token", "participant", "group", "volume_pct", "lvl_highflow"]]

print("Active:")
display(df_active.head(3))
print("Passive:")
display(df_passive.head(3))


In [ ]:
# ==============================================================================
# Clean volume_pct (remove % if present) and convert to float
# ==============================================================================

for df in [df_active, df_passive]:
    df["volume_pct"] = df["volume_pct"] * 100

print("Active  volume_pct range:", df_active["volume_pct"].min(), "-", df_active["volume_pct"].max())
print("Passive volume_pct range:", df_passive["volume_pct"].min(), "-", df_passive["volume_pct"].max())


In [ ]:
# ==============================================================================
# Polynomial regression: volume_pct → volume_db
# ==============================================================================

# Ground truth calibration data
df_clicks = pd.DataFrame({
    "Nb_clicks":  ["Left extreme", "1 click", "2 clicks", "3 clicks", "4 clicks",
                   "5 clicks", "6 clicks", "7 clicks", "8 clicks", "9", "10"],
    "volume_pct": [0, 8, 36, 52, 63, 72, 79, 86, 91, 95, 100],
    "Max_dBSPL":  [58, 58, 59, 60, 65, 71, 72, 75, 81, 84, 86]
})

X      = df_clicks["volume_pct"].values.reshape(-1, 1)
y      = df_clicks["Max_dBSPL"].values
poly   = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
model  = LinearRegression()
model.fit(X_poly, y)
print(f"R² = {model.score(X_poly, y):.4f}")

# Predict for each paradigm
for df in [df_active, df_passive]:
    mask = df["volume_pct"].notna()
    df.loc[mask, "volume_db"] = model.predict(
        poly.transform(df.loc[mask, "volume_pct"].values.reshape(-1, 1))
    ).round(1)

# Plot
x_range      = np.linspace(0, 100, 200).reshape(-1, 1)
y_range_pred = model.predict(poly.transform(x_range))
df_all_vol   = pd.concat([df_active, df_passive])[["volume_pct", "volume_db"]].dropna()

plt.figure(figsize=(10, 6))
plt.scatter(df_clicks["volume_pct"], df_clicks["Max_dBSPL"],
            color='blue', label='Calibration data')
plt.plot(x_range, y_range_pred,
         color='black', linewidth=2, label='Fitted polynomial (degree=2)')
plt.scatter(df_all_vol["volume_pct"], df_all_vol["volume_db"],
            color='red', marker='x', s=80, label='Participant predictions')
plt.xlabel("Volume %")
plt.ylabel("Max dB SPL")
plt.title("Polynomial Regression: Max dB SPL vs Volume %")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Merge prescreening (age, sex, gender) with session data
# ==============================================================================

def build_demographics(df_session, df_prescreening):
    df = df_session.merge(df_prescreening, on="token", how="left")
    df = df.drop(columns=["token"])
    df = df[["participant", "group", "sex", "age", "volume_pct", "volume_db", "lvl_highflow"]]
    return df

df_demo_active  = build_demographics(df_active,  df_pre)
df_demo_passive = build_demographics(df_passive, df_pre)

print(f"Active  demographics: {len(df_demo_active)} participants")
print(f"Passive demographics: {len(df_demo_passive)} participants")
display(df_demo_active.head(3))


In [ ]:
# ==============================================================================
# Sanity checks
# ==============================================================================

for label, df in [("ACTIVE", df_demo_active), ("PASSIVE", df_demo_passive)]:
    print(f"=== {label} ===")
    print(f"  N participants  : {df['participant'].nunique()}")
    print(f"  Missing age     : {df['age'].isna().sum()}")
    print(f"  Missing volume  : {df['volume_db'].isna().sum()}")
    print(f"  Age range       : {df['age'].min():.0f} - {df['age'].max():.0f}")
    print(f"  Volume range dB : {df['volume_db'].min()} - {df['volume_db'].max()}")
    print(f"  Lvl highflow range: {df['lvl_highflow'].min()} - {df['lvl_highflow'].max()}")
    print()


In [ ]:
# ==============================================================================
# Export to NAS
# ==============================================================================

fname_active  = OUT_ACTIVE  / f"{DATE_STR}_active_participants_info.csv"
fname_passive = OUT_PASSIVE / f"{DATE_STR}_passive_participants_info.csv"

df_demo_active.to_csv(fname_active,   index=False)
df_demo_passive.to_csv(fname_passive, index=False)

print(f"Saved -> {fname_active}")
print(f"Saved -> {fname_passive}")
